# RAFT-Stereo corrected fine-tune (DrivingStereo, Phase 1)
Before running: in your Google Drive create folder `raft_finetune/` containing:
- `ds_data.zip`  (the DrivingStereo folder: left/right/disparity)
- `stereo_datasets.py`  (your modified core/stereo_datasets.py)
- `train_stereo.py`  (your modified train_stereo.py with 500-step checkpoints)

Runtime -> Change runtime type -> **T4 GPU**. Then run cells top to bottom.

In [ ]:
!nvidia-smi

Fri Jun 12 17:36:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/raft_finetune'
import os; assert os.path.exists(DRIVE), 'Create raft_finetune/ in Drive and upload the 3 files first'

Mounted at /content/drive


In [ ]:
# clone repo + pretrained models, overlay your two modified files
%cd /content
!git clone -q https://github.com/princeton-vl/RAFT-Stereo.git
%cd /content/RAFT-Stereo
!bash download_models.sh
!cp $DRIVE/stereo_datasets.py core/stereo_datasets.py
!cp $DRIVE/train_stereo.py train_stereo.py
!ls models/ | head

/content
/content/RAFT-Stereo
--2026-06-12 17:36:40--  https://www.dropbox.com/s/ftveifyqcomiwaq/models.zip
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/z720aslw752iypqqdre7c/models.zip?rlkey=3criskutu26wwrx4h9g42aqx0 [following]
--2026-06-12 17:36:40--  https://www.dropbox.com/scl/fi/z720aslw752iypqqdre7c/models.zip?rlkey=3criskutu26wwrx4h9g42aqx0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucb3cb267c1ca965b492853bf486.dl.dropboxusercontent.com/cd/0/inline/DCTyYqSvSm-BmgIjdX9RyahN38QC8XWIdNrz1YXZBkKp_Q0NFS5eEbJ4L-9GeKZgDlJD3BQcqJpCXUFhLdp-waJwo1z0yLUmTeR1txYgLyss0oZ0zLxfDj-doKJrpfdAm0oZngSZxJ_cn_CwI27DBP8U/file# [following]
--2026-06-12 17:36:41--  https://ucb3cb267c1ca965b492853bf486.dl.dropboxuserc

In [ ]:
# unzip data -> datasets/DrivingStereo/{left,right,disparity}
!mkdir -p datasets
!unzip -q $DRIVE/ds_data.zip -d datasets/
!find datasets/DrivingStereo -type f | wc -l   # expect ~3x your pair count

24687


In [ ]:
# holdout split BEFORE training: ~15 frozen eval triples leave the training tree
import os, glob, shutil
root, hold, EVERY_N = 'datasets/DrivingStereo', 'datasets/DS_holdout', 400
lefts = sorted(glob.glob(os.path.join(root, 'left', '*', '*.jpg')))
print(len(lefts), 'left frames')
moved = 0
for lp in lefts[::EVERY_N]:
    seq, fname = os.path.basename(os.path.dirname(lp)), os.path.basename(lp)
    pname = fname.replace('.jpg', '.png')
    rp = os.path.join(root, 'right', seq, fname)
    dp = os.path.join(root, 'disparity', seq, pname)
    if not (os.path.exists(rp) and os.path.exists(dp)):
        continue
    for src, sub, name in ((lp,'left',fname),(rp,'right',fname),(dp,'disparity',pname)):
        d = os.path.join(hold, sub, seq); os.makedirs(d, exist_ok=True)
        shutil.move(src, os.path.join(d, name))
    moved += 1
print('held out', moved, 'triples')

8229 left frames
held out 21 triples


In [ ]:
# checkpoints go STRAIGHT to Drive (disconnect costs nothing)
!mkdir -p $DRIVE/checkpoints
!rm -rf checkpoints && ln -s $DRIVE/checkpoints checkpoints
!ls -la checkpoints

lrwxrwxrwx 1 root root 48 Jun 12 17:38 checkpoints -> /content/drive/MyDrive/raft_finetune/checkpoints


In [ ]:
# TRAIN: corrected config (batch 4, 16 iters, LR 1e-5, 2500 steps, ckpt/500)
!python train_stereo.py --name ds_finetune_v2 \
    --restore_ckpt models/raftstereo-sceneflow.pth \
    --train_datasets drivingstereo \
    --num_steps 2500 --batch_size 4 \
    --train_iters 16 --valid_iters 32 \
    --image_size 288 720 --mixed_precision --lr 0.00001
# If CUDA out-of-memory in the first minute: batch_size 3, or image_size 320 640.

2026-06-12 12:49:42.792722: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Parameter Count: 11116176
2026-06-12 12:49:54,712 INFO     [stereo_datasets.py:287] DrivingStereo: loaded 8208 stereo pairs
2026-06-12 12:49:54,712 INFO     [stereo_datasets.py:338] Adding 8208 samples from DrivingStereo
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_n

In [ ]:
# EVALUATE every checkpoint + pretrained baseline on the holdout (EPE / bad-3)
import sys, glob, argparse, numpy as np, cv2, torch
sys.path.append('core')
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

ARGS = argparse.Namespace(hidden_dims=[128]*3, corr_implementation='reg',
    shared_backbone=False, corr_levels=4, corr_radius=4, n_downsample=2,
    context_norm='batch', slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(ckpt):
    m = torch.nn.DataParallel(RAFTStereo(ARGS), device_ids=[0])
    m.load_state_dict(torch.load(ckpt))
    return m.module.cuda().eval()

def read_img(p):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()

holds = sorted(glob.glob('datasets/DrivingStereo/left/*/*.jpg'))[::400]
ckpts = ['models/raftstereo-sceneflow.pth'] + sorted(glob.glob('checkpoints/*ds_finetune_v2*.pth')) + sorted(glob.glob('checkpoints/*ds_kitti*.pth'))
print(f'{len(holds)} holdout frames, {len(ckpts)} checkpoints\n')

for ck in ckpts:
    model = load_model(ck)
    epes, bad3s = [], []
    with torch.no_grad():
        for lp in holds:
            rp = lp.replace('/left/', '/right/')
            gp = lp.replace('/left/', '/disparity/').replace('.jpg', '.png')
            gt = cv2.imread(gp, cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
            i1, i2 = read_img(lp), read_img(rp)
            padder = InputPadder(i1.shape, divis_by=32)
            i1, i2 = padder.pad(i1, i2)
            _, flow = model(i1, i2, iters=32, test_mode=True)
            pred = np.abs(padder.unpad(flow).cpu().numpy().squeeze())
            v = gt > 0
            err = np.abs(pred[v] - gt[v])
            epes.append(err.mean()); bad3s.append((err > 3.0).mean()*100)
    print(f'{ck:55s}  EPE {np.mean(epes):6.3f}   bad3 {np.mean(bad3s):5.2f}%')

print('\nLower is better. Best checkpoint = lowest EPE that also LOOKS sharp.')

21 holdout frames, 8 checkpoints



/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT-Stereo/core/raft_stereo.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):


models/raftstereo-sceneflow.pth                          EPE  0.859   bad3  2.74%
checkpoints/1000_ds_finetune_v2.pth                      EPE  0.536   bad3  0.68%
checkpoints/1500_ds_finetune_v2.pth                      EPE  0.526   bad3  0.69%
checkpoints/2000_ds_finetune_v2.pth                      EPE  0.515   bad3  0.61%
checkpoints/2500_ds_finetune_v2.pth                      EPE  0.517   bad3  0.65%
checkpoints/500_ds_finetune_v2.pth                       EPE  0.565   bad3  0.89%
checkpoints/ds_finetune_v2.pth                           EPE  0.517   bad3  0.65%
checkpoints/ds_kitti_finetune_v1.pth                     EPE  0.547   bad3  0.76%

Lower is better. Best checkpoint = lowest EPE that also LOOKS sharp.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
labels = ['baseline\nScene Flow', '500', '1000', '1500', '2000', '2500']
epe  = [0.859, 0.565, 0.536, 0.526, 0.515, 0.517]
bad3 = [2.74, 0.89, 0.68, 0.69, 0.61, 0.65]
colors = ['#888888', '#4C72B0', '#4C72B0', '#4C72B0', '#2ca02c', '#4C72B0']
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, vals, title in ((a1, epe, 'EPE (px) — lower is better'), (a2, bad3, 'bad-3 (%) — lower is better')):
    b = ax.bar(labels, vals, color=colors)
    ax.bar_label(b, fmt='%.3f' if ax is a1 else '%.2f', padding=2)
    ax.set_title(title); ax.set_xlabel('fine-tuning step')
fig.suptitle('RAFT-Stereo fine-tune vs pretrained baseline — 21 held-out DrivingStereo frames', y=1.03)
fig.tight_layout()
fig.savefig('finetune_comparison.png', dpi=150, bbox_inches='tight')
!cp finetune_comparison.png $DRIVE/
from google.colab import files; files.download('finetune_comparison.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<Figure size 1200x450 with 2 Axes>

In [ ]:
import os, sys, glob, argparse, numpy as np, cv2, torch
os.chdir('/content/RAFT-Stereo')
sys.path.append('core')
from raft_stereo import RAFTStereo
from utils.utils import InputPadder
import matplotlib.pyplot as plt

ARGS = argparse.Namespace(hidden_dims=[128]*3, corr_implementation='reg',
    shared_backbone=False, corr_levels=4, corr_radius=4, n_downsample=2,
    context_norm='batch', slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(ck):
    m = torch.nn.DataParallel(RAFTStereo(ARGS), device_ids=[0])
    m.load_state_dict(torch.load(ck))
    return m.module.cuda().eval()

def read_img(p):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()

holds = sorted(glob.glob('datasets/DS_holdout/left/*/*.jpg'))
print(len(holds), 'holdout frames found')

lp = holds[5]; rp = lp.replace('/left/', '/right/')
i1, i2 = read_img(lp), read_img(rp)
padder = InputPadder(i1.shape, divis_by=32); i1, i2 = padder.pad(i1, i2)
outs = {}
for name, ck in [('pretrained baseline', 'models/raftstereo-sceneflow.pth'),
                 ('fine-tuned (step 2000)', 'checkpoints/2000_ds_finetune_v2.pth')]:
    m = load_model(ck)
    with torch.no_grad(): _, fl = m(i1, i2, iters=32, test_mode=True)
    outs[name] = np.abs(padder.unpad(fl).cpu().numpy().squeeze())

left = cv2.cvtColor(cv2.imread(lp), cv2.COLOR_BGR2RGB)
vmax = np.percentile(list(outs.values())[0], 98)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(left); axes[0].set_title('left image')
for ax, (name, d) in zip(axes[1:], outs.items()):
    ax.imshow(d, cmap='magma', vmin=0, vmax=vmax); ax.set_title(name)
for ax in axes: ax.axis('off')
fig.tight_layout(); fig.savefig('sharpness_check.png', dpi=150, bbox_inches='tight')
os.system('cp sharpness_check.png /content/drive/MyDrive/raft_finetune/')
from google.colab import files; files.download('sharpness_check.png')

21 holdout frames found


/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT-Stereo/core/raft_stereo.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<Figure size 1800x400 with 3 Axes>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!mkdir -p datasets/KITTI
!unzip -q "/content/drive/MyDrive/raft_finetune/KITTI_training.zip" -d datasets/KITTI/
!ls datasets/KITTI/training

disp_noc_0  disp_occ_0	flow_noc  image_2  obj_map	 viz_flow_occ_dilate_1
disp_noc_1  disp_occ_1	flow_occ  image_3  viz_flow_occ


In [ ]:
path = "core/stereo_datasets.py"
src = open(path).read()
src = src.replace(
    "elif 'kitti' in dataset_name:\n            new_dataset = KITTI(aug_params)",
    "elif 'kitti' in dataset_name:\n            new_dataset = KITTI(aug_params) * 40"
)
open(path, "w").write(src)
!grep -n "KITTI(aug_params)" core/stereo_datasets.py


FileNotFoundError: [Errno 2] No such file or directory: 'core/stereo_datasets.py'

In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
!ls datasets/DrivingStereo

disparity  left  right


All checkpoints are already in your Drive (`raft_finetune/checkpoints/`).
Download the best one to the laptop and compare it visually against the
pretrained model on a familiar frame (sharpness check) before declaring a winner.

In [ ]:
src = open("core/stereo_datasets.py").read()
src = src.replace("new_dataset = KITTI(aug_params)\n", "new_dataset = KITTI(aug_params) * 40\n")
open("core/stereo_datasets.py","w").write(src)
!grep -n "KITTI(aug_params)" core/stereo_datasets.py

334:            new_dataset = KITTI(aug_params) * 40


In [ ]:
!python train_stereo.py --name ds_kitti_finetune_v1 \
  --restore_ckpt models/raftstereo-sceneflow.pth \
  --train_datasets drivingstereo kitti \
  --num_steps 2500 --batch_size 4 \
  --train_iters 16 --valid_iters 32 \
  --image_size 288 720 --mixed_precision --lr 0.00001

2026-06-15 23:56:06.674556: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Parameter Count: 11116176
2026-06-15 23:56:20,106 INFO     [stereo_datasets.py:287] DrivingStereo: loaded 8229 stereo pairs
2026-06-15 23:56:20,106 INFO     [stereo_datasets.py:338] Adding 8229 samples from DrivingStereo
2026-06-15 23:56:20,109 INFO     [stereo_datasets.py:335] Adding 8000 samples from KITTI
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower

In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
import sys; sys.argv = ['x']
import core.stereo_datasets as sd
import argparse
aug = {'crop_size': [288,720], 'min_scale': -0.2, 'max_scale': 0.4, 'do_flip': False, 'yjitter': True}

k = sd.KITTI(aug) * 40
d = sd.DrivingStereo(aug)
print("KITTI samples:", len(k))
print("DrivingStereo samples:", len(d))
print("KITTI fraction of mix: %.0f%%" % (100*len(k)/(len(k)+len(d))))

FileNotFoundError: [Errno 2] No such file or directory: '/content/RAFT-Stereo'

In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
!rm -rf checkpoints
!ln -s /content/drive/MyDrive/raft_finetune/checkpoints checkpoints
!ls checkpoints/

1000_ds_finetune_v2.pth  2000_ds_finetune_v2.pth  500_ds_finetune_v2.pth
1500_ds_finetune_v2.pth  2500_ds_finetune_v2.pth  ds_finetune_v2.pth


In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
!mkdir -p datasets/KITTI
!unzip -q "/content/drive/MyDrive/raft_finetune/KITTI_training.zip" -d datasets/KITTI/
src = open("core/stereo_datasets.py").read()
src = src.replace("new_dataset = KITTI(aug_params)\n", "new_dataset = KITTI(aug_params) * 40\n")
open("core/stereo_datasets.py","w").write(src)
!grep -n "KITTI(aug_params)" core/stereo_datasets.py
!ls datasets/KITTI/training

334:            new_dataset = KITTI(aug_params) * 40
disp_noc_0  disp_occ_0	flow_noc  image_2  obj_map	 viz_flow_occ_dilate_1
disp_noc_1  disp_occ_1	flow_occ  image_3  viz_flow_occ


In [ ]:
!python train_stereo.py --name ds_kitti_finetune_v1 \
  --restore_ckpt models/raftstereo-sceneflow.pth \
  --train_datasets drivingstereo kitti \
  --num_steps 2500 --batch_size 4 \
  --train_iters 16 --valid_iters 32 \
  --image_size 288 720 --mixed_precision --lr 0.00001

2026-06-16 12:59:24.902682: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Parameter Count: 11116176
2026-06-16 12:59:38,632 INFO     [stereo_datasets.py:287] DrivingStereo: loaded 8229 stereo pairs
2026-06-16 12:59:38,632 INFO     [stereo_datasets.py:338] Adding 8229 samples from DrivingStereo
2026-06-16 12:59:38,635 INFO     [stereo_datasets.py:335] Adding 8000 samples from KITTI
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower

In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
exec(open('/content/eval_AB.py').read()) if os.path.exists('/content/eval_AB.py') else print("upload eval_AB.py first")

upload eval_AB.py first


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

models = ['Baseline\n(Scene Flow)', 'Model A\n(DrivingStereo)', 'Model B\n(DS + KITTI)']
epe  = [0.859, 0.515, 0.547]
bad3 = [2.74, 0.61, 0.76]

x = np.arange(len(models))
w = 0.35
fig, ax1 = plt.subplots(figsize=(8, 5))

b1 = ax1.bar(x - w/2, epe, w, label='EPE (px)', color='#534AB7')
ax1.set_ylabel('EPE (px)', color='#534AB7')
ax1.tick_params(axis='y', labelcolor='#534AB7')
ax1.set_ylim(0, 1.0)

ax2 = ax1.twinx()
b2 = ax2.bar(x + w/2, bad3, w, label='bad-3 (%)', color='#1D9E75')
ax2.set_ylabel('bad-3 (%)', color='#1D9E75')
ax2.tick_params(axis='y', labelcolor='#1D9E75')
ax2.set_ylim(0, 3.0)

ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.set_title('RAFT-Stereo fine-tune comparison (DrivingStereo holdout, lower is better)')

for b in b1:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, '{:.3f}'.format(b.get_height()), ha='center', fontsize=9)
for b in b2:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05, '{:.2f}%'.format(b.get_height()), ha='center', fontsize=9)

fig.tight_layout()
plt.savefig('/content/drive/MyDrive/raft_finetune/finetune_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('saved to Drive: finetune_comparison.png')

<Figure size 800x500 with 2 Axes>

saved to Drive: finetune_comparison.png


In [ ]:
import os
os.chdir('/content')
!git clone -q https://github.com/princeton-vl/RAFT-Stereo.git
os.chdir('/content/RAFT-Stereo')
!bash download_models.sh
!cp /content/drive/MyDrive/raft_finetune/stereo_datasets.py core/stereo_datasets.py
!mkdir -p datasets/DrivingStereo checkpoints
!unzip -q "/content/drive/MyDrive/raft_finetune/ds_data.zip" -d datasets/DrivingStereo/
!mv datasets/DrivingStereo/DrivingStereo/* datasets/DrivingStereo/ 2>/dev/null
!rmdir datasets/DrivingStereo/DrivingStereo 2>/dev/null
!cp /content/drive/MyDrive/raft_finetune/checkpoints/2000_ds_finetune_v2.pth checkpoints/
!cp /content/drive/MyDrive/raft_finetune/checkpoints/ds_kitti_finetune_v1.pth checkpoints/
!ls checkpoints/ && ls datasets/DrivingStereo/

--2026-06-17 10:29:20--  https://www.dropbox.com/s/ftveifyqcomiwaq/models.zip
Resolving www.dropbox.com (www.dropbox.com)... 162.125.2.18, 2620:100:6017:18::a27d:212
Connecting to www.dropbox.com (www.dropbox.com)|162.125.2.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/z720aslw752iypqqdre7c/models.zip?rlkey=3criskutu26wwrx4h9g42aqx0 [following]
--2026-06-17 10:29:21--  https://www.dropbox.com/scl/fi/z720aslw752iypqqdre7c/models.zip?rlkey=3criskutu26wwrx4h9g42aqx0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc1d40dc25395b6239270edff0ac.dl.dropboxusercontent.com/cd/0/inline/DCn5wkWDrPR0OQWYLuSGbzhKZ8xccN5R-QnALtw-XSpCKrAy3jDXaozlRs5-WNaGBrGXjvW0zl8RsrfEBGUZlG0-pUTyaxuSP_ntdYbzAkKAeddDzQH2SYT16ffcewxfvCDVp-hW1HzPRHa1wM4A-_rM/file# [following]
--2026-06-17 10:29:21--  https://uc1d40dc25395b6239270edff0ac.dl.dropboxusercontent.com/cd/0/inline/DCn5wkW

In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
print(os.getcwd())
print(os.listdir('.'))
print('--- core ---')
print(os.listdir('core'))

/content/RAFT-Stereo
['models', 'sampler', 'evaluate_stereo.py', 'checkpoints', 'environment_cuda11.yaml', 'environment.yaml', 'README.md', 'download_datasets.sh', 'download_middlebury_2014.sh', 'RAFTStereo.png', '.git', 'core', 'datasets', 'depth_eq.png', 'download_models.sh', 'LICENSE', 'demo.py', 'train_stereo.py']
--- core ---
['extractor.py', 'utils', 'raft_stereo.py', 'update.py', 'stereo_datasets.py', '__init__.py', 'corr.py']


In [ ]:
import os, sys, glob
os.chdir('/content/RAFT-Stereo')
sys.path.insert(0, '/content/RAFT-Stereo/core')
sys.path.insert(0, '/content/RAFT-Stereo')
import numpy as np, cv2, torch
from argparse import Namespace
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

ROOT="datasets/DrivingStereo"; EVERY=400
MODELS={"A_ds_only":"checkpoints/2000_ds_finetune_v2.pth",
        "B_ds_kitti":"checkpoints/ds_kitti_finetune_v1.pth"}
SAVE_VIS=3

ARGS=Namespace(hidden_dims=[128]*3, corr_implementation='reg', shared_backbone=False,
    corr_levels=4, corr_radius=4, n_downsample=2, context_norm='batch',
    slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(ck):
    m=torch.nn.DataParallel(RAFTStereo(ARGS),device_ids=[0])
    m.load_state_dict(torch.load(ck)); return m.module.cuda().eval()

def read_img(p):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()

def holdout():
    out=[]
    for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
        ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
        for i in range(0,len(ls),EVERY):
            n=os.path.splitext(os.path.basename(ls[i]))[0]
            rp=os.path.join(ROOT,'right',seq,n+'.jpg')
            dp=os.path.join(ROOT,'disparity',seq,n+'.png')
            if os.path.exists(rp) and os.path.exists(dp): out.append((ls[i],rp,dp,seq,n))
    return out

@torch.no_grad()
def infer(m,lp,rp):
    a,b=read_img(lp),read_img(rp)
    pad=InputPadder(a.shape,divis_by=32); a,b=pad.pad(a,b)
    _,fl=m(a,b,iters=32,test_mode=True)
    return np.abs(pad.unpad(fl).cpu().numpy().squeeze())

def guided_sharpen(disp, guide_bgr):
    g=cv2.cvtColor(guide_bgr,cv2.COLOR_BGR2GRAY).astype(np.float32)/255.0
    g=cv2.resize(g,(disp.shape[1],disp.shape[0]))
    try:
        import cv2.ximgproc as xi
        return xi.guidedFilter(guide=g.astype(np.float32), src=disp.astype(np.float32), radius=8, eps=1e-4)
    except Exception:
        return cv2.bilateralFilter(disp.astype(np.float32), 9, 50, 9)

def colorize(disp):
    d=np.clip(disp/(disp.max()+1e-6)*255,0,255).astype(np.uint8)
    return cv2.applyColorMap(d, cv2.COLORMAP_MAGMA)

def metrics_split(pred, gt, edge_mask):
    v=gt>0
    if v.sum()==0: return None
    e=np.abs(pred-gt)
    def stat(mask):
        mm=v&mask
        if mm.sum()==0: return (np.nan,np.nan)
        ee=e[mm]; return (float(ee.mean()), float((ee>3).mean()*100))
    return {"all":stat(np.ones_like(v)),"edge":stat(edge_mask),"flat":stat(~edge_mask)}

frames=holdout(); print(f"{len(frames)} holdout frames\n")
for k,ck in MODELS.items():
    if not os.path.exists(ck): print("missing",ck); continue
    m=load_model(ck)
    agg={"raw":[], "sharp":[]}; saved=0
    for lp,rp,dp,seq,n in frames:
        gt=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
        left=cv2.imread(lp)
        pred=infer(m,lp,rp)
        if pred.shape!=gt.shape: pred=cv2.resize(pred,(gt.shape[1],gt.shape[0]))
        gray=cv2.cvtColor(cv2.resize(left,(gt.shape[1],gt.shape[0])),cv2.COLOR_BGR2GRAY)
        gx=cv2.Sobel(gray,cv2.CV_32F,1,0,ksize=3); gy=cv2.Sobel(gray,cv2.CV_32F,0,1,ksize=3)
        grad=np.sqrt(gx*gx+gy*gy); edge=grad>np.percentile(grad,90)
        sharp=guided_sharpen(pred,left)
        agg["raw"].append(metrics_split(pred,gt,edge))
        agg["sharp"].append(metrics_split(sharp,gt,edge))
        if saved<SAVE_VIS:
            L=cv2.resize(left,(gt.shape[1],gt.shape[0]))
            stack=np.vstack([L, colorize(pred), colorize(sharp)])
            cv2.imwrite(f"sharpen_{k}_{saved}.png", stack)
            saved+=1
    def mean(lst,kind,idx):
        vals=[d[kind][idx] for d in lst if d and not np.isnan(d[kind][idx])]
        return np.mean(vals) if vals else float('nan')
    print(f"=== {k} ===")
    print(f"{'':10s} {'allEPE':>8s} {'edgeEPE':>8s} {'flatEPE':>8s} | {'allbad3':>8s} {'edgbad3':>8s} {'fltbad3':>8s}")
    for tag in ["raw","sharp"]:
        print(f"{tag:10s} {mean(agg[tag],'all',0):8.3f} {mean(agg[tag],'edge',0):8.3f} {mean(agg[tag],'flat',0):8.3f} | {mean(agg[tag],'all',1):8.2f} {mean(agg[tag],'edge',1):8.2f} {mean(agg[tag],'flat',1):8.2f}")
    print(f"saved sharpen_{k}_0..{SAVE_VIS-1}.png\n")

print("READ: if 'sharp' edgeEPE < 'raw' edgeEPE -> post-processing recovered the over-smoothed edges.")

22 holdout frames



/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT-Stereo/core/raft_stereo.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):


=== A_ds_only ===
             allEPE  edgeEPE  flatEPE |  allbad3  edgbad3  fltbad3
raw           0.520    0.561    0.519 |     0.75     1.38     0.72
sharp         0.538    0.621    0.534 |     0.98     2.02     0.92
saved sharpen_A_ds_only_0..2.png

=== B_ds_kitti ===
             allEPE  edgeEPE  flatEPE |  allbad3  edgbad3  fltbad3
raw           0.543    0.631    0.539 |     0.78     2.04     0.71
sharp         0.562    0.691    0.555 |     1.02     2.74     0.92
saved sharpen_B_ds_kitti_0..2.png

READ: if 'sharp' edgeEPE < 'raw' edgeEPE -> post-processing recovered the over-smoothed edges.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['Model A\n(DrivingStereo)', 'Model B\n(DS + KITTI)']
edge_epe = [0.561, 0.631]
flat_epe = [0.519, 0.539]

x = np.arange(len(models))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))

b1 = ax.bar(x - w/2, edge_epe, w, label='Edge pixels (thin structures)', color='#C2410C')
b2 = ax.bar(x + w/2, flat_epe, w, label='Flat pixels (road/walls)', color='#534AB7')

ax.set_ylabel('EPE (px)  —  lower is better')
ax.set_title('Over-smoothing: disparity error is higher at thin/edge structures\n(DrivingStereo holdout, 22 frames)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 0.75)

for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                '{:.3f}'.format(b.get_height()), ha='center', fontsize=10)

fig.tight_layout()
plt.savefig('/content/drive/MyDrive/raft_finetune/edge_vs_flat_epe.png', dpi=200, bbox_inches='tight')
plt.show()
print('saved to Drive: edge_vs_flat_epe.png')

<Figure size 800x500 with 1 Axes>

saved to Drive: edge_vs_flat_epe.png


In [ ]:
from IPython.display import Image, display
for i in range(3):
    print(f"--- Model A, frame {i} ---")
    display(Image(f'/content/RAFT-Stereo/sharpen_A_ds_only_{i}.png'))
for i in range(3):
    print(f"--- Model B, frame {i} ---")
    display(Image(f'/content/RAFT-Stereo/sharpen_B_ds_kitti_{i}.png'))

--- Model A, frame 0 ---


<IPython.core.display.Image object>

--- Model A, frame 1 ---


<IPython.core.display.Image object>

--- Model A, frame 2 ---


...

<IPython.core.display.Image object>

--- Model B, frame 1 ---


<IPython.core.display.Image object>

--- Model B, frame 2 ---


<IPython.core.display.Image object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content')
!git clone -q https://github.com/princeton-vl/RAFT-Stereo.git
os.chdir('/content/RAFT-Stereo')
!bash download_models.sh
!mkdir -p checkpoints
!cp /content/drive/MyDrive/raft_finetune/checkpoints/2000_ds_finetune_v2.pth checkpoints/
!ls checkpoints/

Mounted at /content/drive
--2026-06-17 11:57:22--  https://www.dropbox.com/s/ftveifyqcomiwaq/models.zip
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/z720aslw752iypqqdre7c/models.zip?rlkey=3criskutu26wwrx4h9g42aqx0 [following]
--2026-06-17 11:57:22--  https://www.dropbox.com/scl/fi/z720aslw752iypqqdre7c/models.zip?rlkey=3criskutu26wwrx4h9g42aqx0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc6540ff5afcb23322a1c7cc98ee.dl.dropboxusercontent.com/cd/0/inline/DCkGvHwLUsEXofxUx48uNSe77qMO8gibfXgFKI46_T0BC9dqAL81r-DNxJR6z2T_I8XlahDeGSvOqs-1DeA3Gdk2ctEg8SU_kkcT8GIt5_qk2IRGC5aJThItFXIE05hanWeHvAVaIuu9Pxy4axfal428/file# [following]
--2026-06-17 11:57:23--  https://uc6540ff5afcb23322a1c7cc98ee.dl.dropboxuserconte

In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
!mkdir -p datasets/DrivingStereo
!unzip -q "/content/drive/MyDrive/raft_finetune/ds_data.zip" -d datasets/DrivingStereo/
!mv datasets/DrivingStereo/DrivingStereo/* datasets/DrivingStereo/ 2>/dev/null
!rmdir datasets/DrivingStereo/DrivingStereo 2>/dev/null
!cp /content/drive/MyDrive/raft_finetune/stereo_datasets.py core/stereo_datasets.py
print("DrivingStereo:", os.path.exists('datasets/DrivingStereo/left'))

DrivingStereo: True


In [ ]:
import os, sys, glob, time
os.chdir('/content/RAFT-Stereo')
sys.path.insert(0, '/content/RAFT-Stereo')
sys.path.insert(0, '/content/RAFT-Stereo/core')
import numpy as np, cv2, torch
from argparse import Namespace
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

def make_args(corr):
    return Namespace(hidden_dims=[128]*3, corr_implementation=corr, shared_backbone=False,
        corr_levels=4, corr_radius=4, n_downsample=2, context_norm='batch',
        slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(corr):
    m=torch.nn.DataParallel(RAFTStereo(make_args(corr)),device_ids=[0])
    m.load_state_dict(torch.load('checkpoints/2000_ds_finetune_v2.pth'))
    return m.module.cuda().eval()

def read_img(p):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()

# holdout frames
ROOT="datasets/DrivingStereo"
frames=[]
for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
    ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
    for i in range(0,len(ls),400):
        n=os.path.splitext(os.path.basename(ls[i]))[0]
        rp=os.path.join(ROOT,'right',seq,n+'.jpg'); dp=os.path.join(ROOT,'disparity',seq,n+'.png')
        if os.path.exists(rp) and os.path.exists(dp): frames.append((ls[i],rp,dp))
print(f"{len(frames)} holdout frames\n")
print("GPU:", torch.cuda.get_device_name(0), "\n")

print(f"{'corr':5s} {'iters':>5s} | {'ms/frame':>9s} {'FPS':>6s} | {'EPE':>7s} {'bad3%':>7s}")
print("-"*52)
for corr in ['reg','alt']:
    try:
        m=load_model(corr)
    except Exception as e:
        print(f"{corr}: failed to load ({str(e)[:30]}) - skipping"); continue
    for iters in [32,16,8,4]:
        # --- speed (synthetic, just for timing) ---
        a=torch.rand(1,3,416,896).cuda()*255; b=torch.rand(1,3,416,896).cuda()*255
        pad=InputPadder(a.shape,divis_by=32); a,b=pad.pad(a,b)
        with torch.no_grad():
            for _ in range(3): _,_=m(a,b,iters=iters,test_mode=True)
            torch.cuda.synchronize(); t0=time.time(); N=15
            for _ in range(N): _,_=m(a,b,iters=iters,test_mode=True)
            torch.cuda.synchronize(); dt=(time.time()-t0)/N
        # --- accuracy (real holdout) ---
        epes,bad3s=[],[]
        with torch.no_grad():
            for lp,rp,dp in frames:
                gt=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
                i1,i2=read_img(lp),read_img(rp)
                pd=InputPadder(i1.shape,divis_by=32); i1,i2=pd.pad(i1,i2)
                _,fl=m(i1,i2,iters=iters,test_mode=True)
                pred=np.abs(pd.unpad(fl).cpu().numpy().squeeze())
                if pred.shape!=gt.shape: pred=cv2.resize(pred,(gt.shape[1],gt.shape[0]))
                v=gt>0; e=np.abs(pred[v]-gt[v]); epes.append(e.mean()); bad3s.append((e>3).mean()*100)
        print(f"{corr:5s} {iters:5d} | {dt*1000:9.1f} {1/dt:6.2f} | {np.mean(epes):7.3f} {np.mean(bad3s):7.2f}")
    print()
print("Sweet spot = high FPS with EPE still near 0.515 (the 32-iter baseline).")

22 holdout frames

GPU: Tesla T4 

corr  iters |  ms/frame    FPS |     EPE   bad3%
----------------------------------------------------
reg      32 |     556.9   1.80 |   0.520    0.75
reg      16 |     348.3   2.87 |   0.520    0.76
reg       8 |     243.4   4.11 |   0.533    0.81
reg       4 |     192.3   5.20 |   0.569    1.03

alt      32 |    1517.2   0.66 |   0.520    0.75
alt      16 |     844.5   1.18 |   0.520    0.76
alt       8 |     494.2   2.02 |   0.533    0.81
alt       4 |     310.6   3.22 |   0.569    1.04

Sweet spot = high FPS with EPE still near 0.515 (the 32-iter baseline).


In [ ]:
import os, sys, glob, time
os.chdir('/content/RAFT-Stereo')
sys.path.insert(0, '/content/RAFT-Stereo'); sys.path.insert(0, '/content/RAFT-Stereo/core')
import numpy as np, cv2, torch
from argparse import Namespace
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

ARGS=Namespace(hidden_dims=[128]*3, corr_implementation='reg', shared_backbone=False,
    corr_levels=4, corr_radius=4, n_downsample=2, context_norm='batch',
    slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)
m=torch.nn.DataParallel(RAFTStereo(ARGS),device_ids=[0])
m.load_state_dict(torch.load('checkpoints/2000_ds_finetune_v2.pth')); m=m.module.cuda().eval()

ROOT="datasets/DrivingStereo"
frames=[]
for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
    ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
    for i in range(0,len(ls),400):
        n=os.path.splitext(os.path.basename(ls[i]))[0]
        rp=os.path.join(ROOT,'right',seq,n+'.jpg'); dp=os.path.join(ROOT,'disparity',seq,n+'.png')
        if os.path.exists(rp) and os.path.exists(dp): frames.append((ls[i],rp,dp))

def read_scaled(p, scale):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB)
    if scale!=1.0:
        im=cv2.resize(im,(int(im.shape[1]*scale),int(im.shape[0]*scale)),interpolation=cv2.INTER_AREA)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()

print("GPU:", torch.cuda.get_device_name(0))
print(f"\n{'scale':>6s} {'iters':>5s} {'res':>10s} | {'FPS':>6s} | {'EPE':>7s} {'bad3%':>6s}")
print("-"*52)
for scale in [1.0, 0.5, 0.35, 0.25]:
    for iters in [16, 8, 4]:
        # speed (synthetic at this scale)
        H,W=int(400*scale),int(881*scale)
        a=torch.rand(1,3,H,W).cuda()*255; b=torch.rand(1,3,H,W).cuda()*255
        pad=InputPadder(a.shape,divis_by=32); a,b=pad.pad(a,b)
        with torch.no_grad():
            for _ in range(3): _,_=m(a,b,iters=iters,test_mode=True)
            torch.cuda.synchronize(); t0=time.time(); N=15
            for _ in range(N): _,_=m(a,b,iters=iters,test_mode=True)
            torch.cuda.synchronize(); dt=(time.time()-t0)/N
        # accuracy (real frames, scaled, disparity upscaled back & rescaled)
        epes,bad3s=[],[]
        with torch.no_grad():
            for lp,rp,dp in frames:
                gt=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
                i1,i2=read_scaled(lp,scale),read_scaled(rp,scale)
                pd=InputPadder(i1.shape,divis_by=32); i1,i2=pd.pad(i1,i2)
                _,fl=m(i1,i2,iters=iters,test_mode=True)
                pr=np.abs(pd.unpad(fl).cpu().numpy().squeeze())
                # upscale disparity to GT size and scale values by 1/scale (disparity shrinks with downscale)
                pr=cv2.resize(pr,(gt.shape[1],gt.shape[0]))/scale
                v=gt>0; e=np.abs(pr[v]-gt[v]); epes.append(e.mean()); bad3s.append((e>3).mean()*100)
        print(f"{scale:6.2f} {iters:5d} {f'{H}x{W}':>10s} | {1/dt:6.2f} | {np.mean(epes):7.3f} {np.mean(bad3s):6.2f}")
    print()
print("Find the row closest to 30 FPS where EPE is still acceptable (<~0.7).")

GPU: Tesla T4

 scale iters        res |    FPS |     EPE  bad3%
----------------------------------------------------
  1.00    16    400x881 |   3.02 |   0.520   0.76
  1.00     8    400x881 |   4.34 |   0.533   0.81
  1.00     4    400x881 |   5.56 |   0.569   1.03

  0.50    16    200x440 |   8.21 |   0.784   2.16
  0.50     8    200x440 |  13.24 |   0.814   2.41
  0.50     4    200x440 |  16.96 |   0.911   3.16

  0.35    16    140x308 |   9.18 |   1.239   5.62
  0.35     8    140x308 |  18.80 |   1.297   6.49
  0.35     4    140x308 |  28.72 |   1.613  10.68

  0.25    16    100x220 |  11.67 |   1.941  16.08
  0.25     8    100x220 |  15.83 |   1.995  17.13
  0.25     4    100x220 |  28.86 |   2.435  23.69

Find the row closest to 30 FPS where EPE is still acceptable (<~0.7).


In [ ]:
import matplotlib.pyplot as plt

# data from the sweep (T4)
data = [
    # scale, iters, FPS, EPE
    (1.00, 16, 3.02, 0.520),
    (1.00,  8, 4.34, 0.533),
    (1.00,  4, 5.56, 0.569),
    (0.50, 16, 8.21, 0.784),
    (0.50,  8, 13.24, 0.814),
    (0.50,  4, 16.96, 0.911),
    (0.35, 16, 9.18, 1.239),
    (0.35,  8, 18.80, 1.297),
    (0.35,  4, 28.72, 1.613),
    (0.25, 16, 11.67, 1.941),
    (0.25,  8, 15.83, 1.995),
    (0.25,  4, 28.86, 2.435),
]

# color by resolution
colors = {1.00:'#1D9E75', 0.50:'#534AB7', 0.35:'#C2410C', 0.25:'#9CA3AF'}
labels_done = set()

fig, ax = plt.subplots(figsize=(9, 6))
for scale, iters, fps, epe in data:
    lab = f'{int(scale*100)}% resolution' if scale not in labels_done else None
    labels_done.add(scale)
    ax.scatter(fps, epe, s=120, color=colors[scale], label=lab, zorder=3, edgecolors='white', linewidth=1)
    ax.annotate(f'{iters}it', (fps, epe), textcoords="offset points", xytext=(6,5), fontsize=8)

# 30 FPS real-time line
ax.axvline(30, color='red', linestyle='--', alpha=0.6, zorder=1)
ax.text(30, ax.get_ylim()[1]*0.95, ' 30 FPS (real-time)', color='red', fontsize=9, va='top')

# baseline accuracy line
ax.axhline(0.520, color='green', linestyle=':', alpha=0.5, zorder=1)
ax.text(ax.get_xlim()[1]*0.6, 0.520, 'baseline accuracy (0.52)', color='green', fontsize=8, va='bottom')

# highlight the sweet spot
ax.annotate('sweet spot\n13 FPS, EPE 0.81', xy=(13.24, 0.814), xytext=(15, 0.55),
            fontsize=9, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlabel('Speed (FPS)  →  faster')
ax.set_ylabel('EPE (px)  →  worse')
ax.set_title('RAFT-Stereo speed vs accuracy tradeoff (Tesla T4)\nlabels = iterations; color = input resolution')
ax.legend(loc='upper right')
ax.grid(alpha=0.2, zorder=0)

fig.tight_layout()
plt.savefig('/content/drive/MyDrive/raft_finetune/speed_accuracy_tradeoff.png', dpi=200, bbox_inches='tight')
plt.show()
print('saved to Drive: speed_accuracy_tradeoff.png')

<Figure size 900x600 with 1 Axes>

saved to Drive: speed_accuracy_tradeoff.png


In [ ]:
import os, sys, glob, time
os.chdir('/content/RAFT-Stereo')
import numpy as np, cv2

ROOT="datasets/DrivingStereo"
frames=[]
for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
    ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
    for i in range(0,len(ls),400):
        n=os.path.splitext(os.path.basename(ls[i]))[0]
        rp=os.path.join(ROOT,'right',seq,n+'.jpg'); dp=os.path.join(ROOT,'disparity',seq,n+'.png')
        if os.path.exists(rp) and os.path.exists(dp): frames.append((ls[i],rp,dp))
print(f"{len(frames)} holdout frames\n")

# SGBM setup (tuned reasonably for ~880-wide automotive stereo)
num_disp = 192   # must be divisible by 16; covers the disparity range
block = 5
sgbm = cv2.StereoSGBM_create(
    minDisparity=0, numDisparities=num_disp, blockSize=block,
    P1=8*3*block**2, P2=32*3*block**2,
    disp12MaxDiff=1, uniquenessRatio=10, speckleWindowSize=100, speckleRange=2,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)

epes, bad3s, times = [], [], []
saved=0
for lp,rp,dp in frames:
    l=cv2.imread(lp); r=cv2.imread(rp)
    gt=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
    lg=cv2.cvtColor(l,cv2.COLOR_BGR2GRAY); rg=cv2.cvtColor(r,cv2.COLOR_BGR2GRAY)
    t0=time.time()
    disp=sgbm.compute(lg,rg).astype(np.float32)/16.0   # SGBM returns disp*16
    times.append(time.time()-t0)
    v=(gt>0)&(disp>0)   # SGBM has invalid (<=0) regions; only score where both valid
    if v.sum()==0: continue
    e=np.abs(disp[v]-gt[v]); epes.append(e.mean()); bad3s.append((e>3).mean()*100)
    if saved<3:
        d=np.clip(disp/disp.max()*255,0,255).astype(np.uint8)
        vis=cv2.applyColorMap(d,cv2.COLORMAP_MAGMA)
        stack=np.vstack([l, vis])
        cv2.imwrite(f"sgbm_{saved}.png", stack); saved+=1

print("=== SGBM (classical) ===")
print(f"EPE:   {np.mean(epes):.3f}")
print(f"bad3:  {np.mean(bad3s):.2f}%")
print(f"speed: {np.mean(times)*1000:.1f} ms/frame  ({1/np.mean(times):.1f} FPS, CPU)")
print(f"valid-pixel coverage: {'(SGBM leaves holes where matching fails)'}")
print("\n--- vs RAFT-Stereo (Model A, 16 iters) ---")
print("RAFT-Stereo:  EPE 0.520,  bad3 0.76%")
print("saved sgbm_0..2.png (top=left image, bottom=SGBM disparity)")

22 holdout frames

=== SGBM (classical) ===
EPE:   1.040
bad3:  3.16%
speed: 128.2 ms/frame  (7.8 FPS, CPU)
valid-pixel coverage: (SGBM leaves holes where matching fails)

--- vs RAFT-Stereo (Model A, 16 iters) ---
RAFT-Stereo:  EPE 0.520,  bad3 0.76%
saved sgbm_0..2.png (top=left image, bottom=SGBM disparity)


In [ ]:
from IPython.display import Image, display
for i in range(3): display(Image(f'/content/RAFT-Stereo/sgbm_{i}.png'))

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

Mounted at /content/drive
repo: True
data: False
ckpt: True
setup done

fx=1003.556, baseline=1.0892 m, depth = 1093.1/disparity



FileNotFoundError: [Errno 2] No such file or directory: 'datasets/DrivingStereo/left'

In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
os.makedirs('datasets/DrivingStereo', exist_ok=True)
# unzip fresh
os.system('unzip -o -q "/content/drive/MyDrive/raft_finetune/ds_data.zip" -d datasets/DrivingStereo/')
# see what landed
print("contents of datasets/DrivingStereo:")
print(os.listdir('datasets/DrivingStereo'))
# if nested, show one level deeper
for d in os.listdir('datasets/DrivingStereo'):
    p = f'datasets/DrivingStereo/{d}'
    if os.path.isdir(p):
        print(f"  {d}/ ->", os.listdir(p)[:5])

contents of datasets/DrivingStereo:
['DrivingStereo']
  DrivingStereo/ -> ['left', 'right', 'disparity']


In [ ]:
import os
os.chdir('/content/RAFT-Stereo')

# fix the double-nesting: move inner DrivingStereo/* up one level
if os.path.exists('datasets/DrivingStereo/DrivingStereo/left'):
    os.system('mv datasets/DrivingStereo/DrivingStereo/left datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/right datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/disparity datasets/DrivingStereo/')
    os.system('rmdir datasets/DrivingStereo/DrivingStereo')
print("data at right place:", os.path.exists('datasets/DrivingStereo/left'))

# ============ 3D VALIDATION ============
import sys, glob
sys.path.insert(0, '/content/RAFT-Stereo'); sys.path.insert(0, '/content/RAFT-Stereo/core')
import numpy as np, cv2, torch
from argparse import Namespace
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

# DrivingStereo half-res calibration (from your calib file)
FX = 1003.556
CX = 455.689
CY = 197.663
BASELINE = 0.5446          # = 1093.101 / 1003.556, from P_rect_103
print(f"fx={FX}, baseline={BASELINE} m, depth = {FX*BASELINE:.1f}/disparity\n")

ROOT="datasets/DrivingStereo"; EVERY=400
ARGS=Namespace(hidden_dims=[128]*3, corr_implementation='reg', shared_backbone=False,
    corr_levels=4, corr_radius=4, n_downsample=2, context_norm='batch',
    slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(ck):
    m=torch.nn.DataParallel(RAFTStereo(ARGS),device_ids=[0])
    m.load_state_dict(torch.load(ck)); return m.module.cuda().eval()

def read_img(p):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()

def holdout():
    out=[]
    for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
        ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
        for i in range(0,len(ls),EVERY):
            n=os.path.splitext(os.path.basename(ls[i]))[0]
            rp=os.path.join(ROOT,'right',seq,n+'.jpg'); dp=os.path.join(ROOT,'disparity',seq,n+'.png')
            if os.path.exists(rp) and os.path.exists(dp): out.append((ls[i],rp,dp))
    return out

def disp_to_3d(disp):
    h,w=disp.shape; valid=disp>0.5
    Z=np.zeros_like(disp); Z[valid]=FX*BASELINE/disp[valid]
    ys,xs=np.mgrid[0:h,0:w]
    X=(xs-CX)*Z/FX; Y=(ys-CY)*Z/FX
    return X,Y,Z,valid

@torch.no_grad()
def infer(m,lp,rp):
    a,b=read_img(lp),read_img(rp)
    pad=InputPadder(a.shape,divis_by=32); a,b=pad.pad(a,b)
    _,fl=m(a,b,iters=16,test_mode=True)
    return np.abs(pad.unpad(fl).cpu().numpy().squeeze())

frames=holdout(); print(f"{len(frames)} holdout frames\n")
m=load_model('checkpoints/2000_ds_finetune_v2.pth')

all_err, edge_err, depth_err = [], [], []
for lp,rp,dp in frames:
    gt_disp=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
    pred=infer(m,lp,rp)
    if pred.shape!=gt_disp.shape: pred=cv2.resize(pred,(gt_disp.shape[1],gt_disp.shape[0]))
    Xp,Yp,Zp,vp=disp_to_3d(pred); Xg,Yg,Zg,vg=disp_to_3d(gt_disp)
    v=vp&vg&(Zg>0.5)&(Zg<80)
    d3=np.sqrt((Xp-Xg)**2+(Yp-Yg)**2+(Zp-Zg)**2)
    all_err.append(np.median(d3[v]))
    depth_err.append(np.median(np.abs(Zp[v]-Zg[v])))
    gray=cv2.cvtColor(cv2.resize(cv2.imread(lp),(gt_disp.shape[1],gt_disp.shape[0])),cv2.COLOR_BGR2GRAY)
    gx=cv2.Sobel(gray,cv2.CV_32F,1,0,3); gy=cv2.Sobel(gray,cv2.CV_32F,0,1,3)
    grad=np.sqrt(gx*gx+gy*gy); edge=grad>np.percentile(grad,90); ev=v&edge
    if ev.sum()>0: edge_err.append(np.median(d3[ev]))

print("=== 3D coordinate accuracy (camera vs ground truth) ===")
print(f"median 3D error (all pixels):   {np.mean(all_err)*100:.1f} cm")
print(f"median 3D error (edge pixels):  {np.mean(edge_err)*100:.1f} cm")
print(f"median depth-only error (Z):    {np.mean(depth_err)*100:.1f} cm")
print(f"\n(median over {len(frames)} frames; 3D = Euclidean XYZ distance to GT-disparity 3D points)")

data at right place: True
fx=1003.556, baseline=0.5446 m, depth = 546.5/disparity

22 holdout frames



RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select 2000_ds_finetune_v2.pth from your laptop's Downloads

Saving 2000_ds_finetune_v2.pth to 2000_ds_finetune_v2.pth


In [ ]:
import os
os.chdir('/content/RAFT-Stereo')

# fix the double-nesting: move inner DrivingStereo/* up one level
if os.path.exists('datasets/DrivingStereo/DrivingStereo/left'):
    os.system('mv datasets/DrivingStereo/DrivingStereo/left datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/right datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/disparity datasets/DrivingStereo/')
    os.system('rmdir datasets/DrivingStereo/DrivingStereo')
print("data at right place:", os.path.exists('datasets/DrivingStereo/left'))

# ============ 3D VALIDATION ============
import sys, glob
sys.path.insert(0, '/content/RAFT-Stereo'); sys.path.insert(0, '/content/RAFT-Stereo/core')
import numpy as np, cv2, torch
from argparse import Namespace
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

# DrivingStereo half-res calibration (from your calib file)
FX = 1003.556
CX = 455.689
CY = 197.663
BASELINE = 0.5446          # = 1093.101 / 1003.556, from P_rect_103
print(f"fx={FX}, baseline={BASELINE} m, depth = {FX*BASELINE:.1f}/disparity\n")

ROOT="datasets/DrivingStereo"; EVERY=400
ARGS=Namespace(hidden_dims=[128]*3, corr_implementation='reg', shared_backbone=False,
    corr_levels=4, corr_radius=4, n_downsample=2, context_norm='batch',
    slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(ck):
    m=torch.nn.DataParallel(RAFTStereo(ARGS),device_ids=[0])
    m.load_state_dict(torch.load(ck)); return m.module.cuda().eval()

def read_img(p):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()

def holdout():
    out=[]
    for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
        ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
        for i in range(0,len(ls),EVERY):
            n=os.path.splitext(os.path.basename(ls[i]))[0]
            rp=os.path.join(ROOT,'right',seq,n+'.jpg'); dp=os.path.join(ROOT,'disparity',seq,n+'.png')
            if os.path.exists(rp) and os.path.exists(dp): out.append((ls[i],rp,dp))
    return out

def disp_to_3d(disp):
    h,w=disp.shape; valid=disp>0.5
    Z=np.zeros_like(disp); Z[valid]=FX*BASELINE/disp[valid]
    ys,xs=np.mgrid[0:h,0:w]
    X=(xs-CX)*Z/FX; Y=(ys-CY)*Z/FX
    return X,Y,Z,valid

@torch.no_grad()
def infer(m,lp,rp):
    a,b=read_img(lp),read_img(rp)
    pad=InputPadder(a.shape,divis_by=32); a,b=pad.pad(a,b)
    _,fl=m(a,b,iters=16,test_mode=True)
    return np.abs(pad.unpad(fl).cpu().numpy().squeeze())

frames=holdout(); print(f"{len(frames)} holdout frames\n")
m=load_model('checkpoints/2000_ds_finetune_v2.pth')

all_err, edge_err, depth_err = [], [], []
for lp,rp,dp in frames:
    gt_disp=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
    pred=infer(m,lp,rp)
    if pred.shape!=gt_disp.shape: pred=cv2.resize(pred,(gt_disp.shape[1],gt_disp.shape[0]))
    Xp,Yp,Zp,vp=disp_to_3d(pred); Xg,Yg,Zg,vg=disp_to_3d(gt_disp)
    v=vp&vg&(Zg>0.5)&(Zg<80)
    d3=np.sqrt((Xp-Xg)**2+(Yp-Yg)**2+(Zp-Zg)**2)
    all_err.append(np.median(d3[v]))
    depth_err.append(np.median(np.abs(Zp[v]-Zg[v])))
    gray=cv2.cvtColor(cv2.resize(cv2.imread(lp),(gt_disp.shape[1],gt_disp.shape[0])),cv2.COLOR_BGR2GRAY)
    gx=cv2.Sobel(gray,cv2.CV_32F,1,0,3); gy=cv2.Sobel(gray,cv2.CV_32F,0,1,3)
    grad=np.sqrt(gx*gx+gy*gy); edge=grad>np.percentile(grad,90); ev=v&edge
    if ev.sum()>0: edge_err.append(np.median(d3[ev]))

print("=== 3D coordinate accuracy (camera vs ground truth) ===")
print(f"median 3D error (all pixels):   {np.mean(all_err)*100:.1f} cm")
print(f"median 3D error (edge pixels):  {np.mean(edge_err)*100:.1f} cm")
print(f"median depth-only error (Z):    {np.mean(depth_err)*100:.1f} cm")
print(f"\n(median over {len(frames)} frames; 3D = Euclidean XYZ distance to GT-disparity 3D points)")

data at right place: True
fx=1003.556, baseline=0.5446 m, depth = 546.5/disparity

22 holdout frames



/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT-Stereo/core/raft_stereo.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):


=== 3D coordinate accuracy (camera vs ground truth) ===
median 3D error (all pixels):   21.9 cm
median 3D error (edge pixels):  39.5 cm
median depth-only error (Z):    21.2 cm

(median over 22 frames; 3D = Euclidean XYZ distance to GT-disparity 3D points)


In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
print("repo:", os.path.exists('core/raft_stereo.py'))
print("checkpoint:", os.path.exists('checkpoints/2000_ds_finetune_v2.pth'))
print("data:", os.path.exists('datasets/DrivingStereo/left'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/RAFT-Stereo'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/raft_finetune'
import os; assert os.path.exists(DRIVE), 'raft_finetune not found in Drive'
print("Drive OK:", os.listdir(DRIVE))

Mounted at /content/drive
Drive OK: ['stereo_datasets.py', 'train_stereo.py', 'ds_data.zip', 'checkpoints', 'sharpness_check.png', 'KITTI_training.zip', 'finetune_comparison.png', 'edge_vs_flat_epe.png', 'speed_accuracy_tradeoff.png']


In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
if os.path.exists('datasets/DrivingStereo/DrivingStereo/left'):
    os.system('mv datasets/DrivingStereo/DrivingStereo/left datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/right datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/disparity datasets/DrivingStereo/')
    os.system('rmdir datasets/DrivingStereo/DrivingStereo')

import sys, glob
sys.path.insert(0, '/content/RAFT-Stereo'); sys.path.insert(0, '/content/RAFT-Stereo/core')
import numpy as np, cv2, torch
from argparse import Namespace
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

FX = 1003.556; CX = 455.689; CY = 197.663; BASELINE = 0.5446
print(f"fx={FX}, baseline={BASELINE} m, depth = {FX*BASELINE:.1f}/disparity\n")

ROOT="datasets/DrivingStereo"; EVERY=400
ARGS=Namespace(hidden_dims=[128]*3, corr_implementation='reg', shared_backbone=False,
    corr_levels=4, corr_radius=4, n_downsample=2, context_norm='batch',
    slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(ck):
    m=torch.nn.DataParallel(RAFTStereo(ARGS),device_ids=[0])
    m.load_state_dict(torch.load(ck)); return m.module.cuda().eval()
def read_img(p):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()
def holdout():
    out=[]
    for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
        ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
        for i in range(0,len(ls),EVERY):
            n=os.path.splitext(os.path.basename(ls[i]))[0]
            rp=os.path.join(ROOT,'right',seq,n+'.jpg'); dp=os.path.join(ROOT,'disparity',seq,n+'.png')
            if os.path.exists(rp) and os.path.exists(dp): out.append((ls[i],rp,dp))
    return out
def disp_to_3d(disp):
    h,w=disp.shape; valid=disp>0.5
    Z=np.zeros_like(disp); Z[valid]=FX*BASELINE/disp[valid]
    ys,xs=np.mgrid[0:h,0:w]
    X=(xs-CX)*Z/FX; Y=(ys-CY)*Z/FX
    return X,Y,Z,valid
@torch.no_grad()
def infer(m,lp,rp):
    a,b=read_img(lp),read_img(rp)
    pad=InputPadder(a.shape,divis_by=32); a,b=pad.pad(a,b)
    _,fl=m(a,b,iters=16,test_mode=True)
    return np.abs(pad.unpad(fl).cpu().numpy().squeeze())

frames=holdout(); print(f"{len(frames)} holdout frames\n")
m=load_model('checkpoints/2000_ds_finetune_v2.pth')

all_err, edge_err, depth_err = [], [], []
all_depths = []                                          # <-- ADDED
for lp,rp,dp in frames:
    gt_disp=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
    pred=infer(m,lp,rp)
    if pred.shape!=gt_disp.shape: pred=cv2.resize(pred,(gt_disp.shape[1],gt_disp.shape[0]))
    Xp,Yp,Zp,vp=disp_to_3d(pred); Xg,Yg,Zg,vg=disp_to_3d(gt_disp)
    v=vp&vg&(Zg>0.5)&(Zg<80)
    d3=np.sqrt((Xp-Xg)**2+(Yp-Yg)**2+(Zp-Zg)**2)
    all_err.append(np.median(d3[v]))
    depth_err.append(np.median(np.abs(Zp[v]-Zg[v])))
    all_depths.append(np.median(Zg[v]))                  # <-- ADDED
    gray=cv2.cvtColor(cv2.resize(cv2.imread(lp),(gt_disp.shape[1],gt_disp.shape[0])),cv2.COLOR_BGR2GRAY)
    gx=cv2.Sobel(gray,cv2.CV_32F,1,0,3); gy=cv2.Sobel(gray,cv2.CV_32F,0,1,3)
    grad=np.sqrt(gx*gx+gy*gy); edge=grad>np.percentile(grad,90); ev=v&edge
    if ev.sum()>0: edge_err.append(np.median(d3[ev]))

print("=== 3D coordinate accuracy (camera vs ground truth) ===")
print(f"median 3D error (all pixels):   {np.mean(all_err)*100:.1f} cm")
print(f"median 3D error (edge pixels):  {np.mean(edge_err)*100:.1f} cm")
print(f"median depth-only error (Z):    {np.mean(depth_err)*100:.1f} cm")
print(f"median GT holdout depth:        {np.median(all_depths):.1f} m   "      # <-- ADDED
      f"(range {np.min(all_depths):.1f}-{np.max(all_depths):.1f} m)")          # <-- ADDED
print(f"\n(median over {len(frames)} frames)")

fx=1003.556, baseline=0.5446 m, depth = 546.5/disparity

22 holdout frames



/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT-Stereo/core/raft_stereo.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):


=== 3D coordinate accuracy (camera vs ground truth) ===
median 3D error (all pixels):   21.9 cm
median 3D error (edge pixels):  39.5 cm
median depth-only error (Z):    21.2 cm
median GT holdout depth:        16.2 m   (range 12.3-22.4 m)

(median over 22 frames)


In [ ]:
import os
os.chdir('/content/RAFT-Stereo')
if os.path.exists('datasets/DrivingStereo/DrivingStereo/left'):
    os.system('mv datasets/DrivingStereo/DrivingStereo/left datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/right datasets/DrivingStereo/')
    os.system('mv datasets/DrivingStereo/DrivingStereo/disparity datasets/DrivingStereo/')
    os.system('rmdir datasets/DrivingStereo/DrivingStereo')

import sys, glob
sys.path.insert(0, '/content/RAFT-Stereo'); sys.path.insert(0, '/content/RAFT-Stereo/core')
import numpy as np, cv2, torch
from argparse import Namespace
from raft_stereo import RAFTStereo
from utils.utils import InputPadder

FX = 1003.556; CX = 455.689; CY = 197.663; BASELINE = 0.5446
print(f"fx={FX}, baseline={BASELINE} m, depth = {FX*BASELINE:.1f}/disparity\n")

ROOT="datasets/DrivingStereo"; EVERY=400
ARGS=Namespace(hidden_dims=[128]*3, corr_implementation='reg', shared_backbone=False,
    corr_levels=4, corr_radius=4, n_downsample=2, context_norm='batch',
    slow_fast_gru=False, n_gru_layers=3, mixed_precision=True)

def load_model(ck):
    m=torch.nn.DataParallel(RAFTStereo(ARGS),device_ids=[0])
    m.load_state_dict(torch.load(ck)); return m.module.cuda().eval()
def read_img(p):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(im).permute(2,0,1).float()[None].cuda()
def holdout():
    out=[]
    for seq in sorted(os.listdir(os.path.join(ROOT,'left'))):
        ls=sorted(glob.glob(os.path.join(ROOT,'left',seq,'*.jpg')))
        for i in range(0,len(ls),EVERY):
            n=os.path.splitext(os.path.basename(ls[i]))[0]
            rp=os.path.join(ROOT,'right',seq,n+'.jpg'); dp=os.path.join(ROOT,'disparity',seq,n+'.png')
            if os.path.exists(rp) and os.path.exists(dp): out.append((ls[i],rp,dp))
    return out
def disp_to_3d(disp):
    h,w=disp.shape; valid=disp>0.5
    Z=np.zeros_like(disp); Z[valid]=FX*BASELINE/disp[valid]
    ys,xs=np.mgrid[0:h,0:w]
    X=(xs-CX)*Z/FX; Y=(ys-CY)*Z/FX
    return X,Y,Z,valid
@torch.no_grad()
def infer(m,lp,rp):
    a,b=read_img(lp),read_img(rp)
    pad=InputPadder(a.shape,divis_by=32); a,b=pad.pad(a,b)
    _,fl=m(a,b,iters=16,test_mode=True)
    return np.abs(pad.unpad(fl).cpu().numpy().squeeze())

frames=holdout(); print(f"{len(frames)} holdout frames\n")
m=load_model('checkpoints/2000_ds_finetune_v2.pth')

all_err, edge_err, depth_err = [], [], []
all_depths = []                                          # <-- ADDED
for lp,rp,dp in frames:
    gt_disp=cv2.imread(dp,cv2.IMREAD_UNCHANGED).astype(np.float32)/256.0
    pred=infer(m,lp,rp)
    if pred.shape!=gt_disp.shape: pred=cv2.resize(pred,(gt_disp.shape[1],gt_disp.shape[0]))
    Xp,Yp,Zp,vp=disp_to_3d(pred); Xg,Yg,Zg,vg=disp_to_3d(gt_disp)
    v=vp&vg&(Zg>0.5)&(Zg<80)
    d3=np.sqrt((Xp-Xg)**2+(Yp-Yg)**2+(Zp-Zg)**2)
    all_err.append(np.median(d3[v]))
    depth_err.append(np.median(np.abs(Zp[v]-Zg[v])))
    all_depths.append(np.median(Zg[v]))                  # <-- ADDED
    gray=cv2.cvtColor(cv2.resize(cv2.imread(lp),(gt_disp.shape[1],gt_disp.shape[0])),cv2.COLOR_BGR2GRAY)
    gx=cv2.Sobel(gray,cv2.CV_32F,1,0,3); gy=cv2.Sobel(gray,cv2.CV_32F,0,1,3)
    grad=np.sqrt(gx*gx+gy*gy); edge=grad>np.percentile(grad,90); ev=v&edge
    if ev.sum()>0: edge_err.append(np.median(d3[ev]))

print("=== 3D coordinate accuracy (camera vs ground truth) ===")
print(f"median 3D error (all pixels):   {np.mean(all_err)*100:.1f} cm")
print(f"median 3D error (edge pixels):  {np.mean(edge_err)*100:.1f} cm")
print(f"median depth-only error (Z):    {np.mean(depth_err)*100:.1f} cm")
print(f"median GT holdout depth:        {np.median(all_depths):.1f} m   "      # <-- ADDED
      f"(range {np.min(all_depths):.1f}-{np.max(all_depths):.1f} m)")          # <-- ADDED
print(f"\n(median over {len(frames)} frames)")

fx=1003.556, baseline=0.5446 m, depth = 546.5/disparity

22 holdout frames



/content/RAFT-Stereo/core/raft_stereo.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT-Stereo/core/raft_stereo.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):


=== 3D coordinate accuracy (camera vs ground truth) ===
median 3D error (all pixels):   21.9 cm
median 3D error (edge pixels):  39.5 cm
median depth-only error (Z):    21.2 cm
median GT holdout depth:        16.2 m   (range 12.3-22.4 m)

(median over 22 frames)
